# Embedding Techniques

**Module:** 01 — Embeddings

From Word2Vec to modern sentence transformers, embedding techniques evolved from static word tables to contextual, instruction-aware encoders.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain CBOW, Skip-Gram, GloVe, and FastText ideas
- Contrast static word vectors with contextual BERT embeddings
- Use sentence-transformer patterns for retrieval embeddings
- Decide when classical vs transformer embeddings are enough


## Word2Vec — CBOW & Skip-Gram

**Definition.** Word2Vec learns static word vectors via local context prediction: **CBOW** predicts a word from context; **Skip-Gram** predicts context from a word.

**Why it matters.** First widely used method that made distributional semantics practical and fast.

**How it works.** Shallow network + negative sampling on a large corpus; each word gets one vector.

**Intuition.** You shall know a word by the company it keeps (Firth).

**Common pitfalls.**
- One vector per word fails on polysemy ('bank' river vs finance)
- OOV words without subwords

**When to use.** Teaching, lightweight classical NLP—not modern RAG retrieval.

```mermaid
flowchart LR
  subgraph CBOW
    c1[context] --> p1[predict center]
  end
  subgraph SkipGram
    w[center word] --> p2[predict context]
  end
```


In [ ]:
# Skip-gram training pair generator (educational)
def skipgram_pairs(tokens, window=2):
    pairs = []
    for i, center in enumerate(tokens):
        for j in range(max(0, i - window), min(len(tokens), i + window + 1)):
            if i == j:
                continue
            pairs.append((center, tokens[j]))
    return pairs

print(skipgram_pairs("the cat sat on the mat".split(), window=1)[:8])


In [ ]:
# Tiny CBOW-style context → target dataset
def cbow_examples(tokens, window=2):
    data = []
    for i in range(len(tokens)):
        left = tokens[max(0, i-window):i]
        right = tokens[i+1:i+window+1]
        data.append((left + right, tokens[i]))
    return data

for ctx, tgt in cbow_examples("refunds arrive within days".split(), 1):
    print(ctx, "→", tgt)


## GloVe

**Definition.** GloVe factorizes a global word-word co-occurrence matrix into dense vectors.

**Why it matters.** Combines corpus statistics globally rather than only local windows.

**How it works.** Weighted least squares on log co-occurrence counts → word vectors.

**Intuition.** If words often appear together in the whole corpus, pull them closer.

**Common pitfalls.**
- Still static / non-contextual
- Heavy memory for huge vocab matrices

**When to use.** Historical baselines and analysis—not new production RAG systems.


In [ ]:
# Co-occurrence matrix sketch
from collections import Counter
import itertools

tokens = "payment refund refund policy shipping policy".split()
window = 1
counts = Counter()
for i, w in enumerate(tokens):
    for j in range(max(0, i-window), min(len(tokens), i+window+1)):
        if i == j: continue
        counts[(w, tokens[j])] += 1
print(counts.most_common(6))


## FastText

**Definition.** FastText extends Word2Vec with **subword n-grams**, so rare/OOV words get vectors from their pieces.

**Why it matters.** Robust to typos, morphologically rich languages, and rare terms.

**How it works.** Represent a word as the sum of its character n-gram vectors plus the word itself.

**Intuition.** If you know 'embed' and 'ing', you can invent 'embedding'.

**Common pitfalls.**
- Still largely non-contextual across sentences

**When to use.** Classical pipelines needing OOV robustness; multilingual baselines.


In [ ]:
def char_ngrams(word, n=3):
    wrapped = f"<{word}>"
    return [wrapped[i:i+n] for i in range(len(wrapped)-n+1)]

print(char_ngrams("refund"))
print("OOV 'refunds' shares ngrams:", set(char_ngrams("refund")) & set(char_ngrams("refunds")))


## BERT Embeddings

**Definition.** BERT produces **contextual** token embeddings with a bidirectional transformer encoder pretrained via masked language modeling.

**Why it matters.** Word meaning depends on context—'bank' in river vs finance differs.

**How it works.** Tokenize → transformer layers → per-token hidden states. For sentence vectors, pool (CLS/mean). Vanilla BERT is not trained as a retrieval similarity model.

**Intuition.** The same ID card changes outfits depending on the party (sentence).

**Common pitfalls.**
- Using raw BERT CLS vectors for cosine search without fine-tuning (often weak)
- Ignoring WordPiece tokenization quirks

**When to use.** As a base encoder; prefer sentence-transformer / embedding models for retrieval.


In [ ]:
# Contextuality toy: same word, different average context features
import numpy as np

def ctx_embed(sentence, target):
    # Fake: hash of neighbors influences vector
    toks = sentence.lower().split()
    idx = toks.index(target)
    neigh = toks[max(0, idx-1):idx] + toks[idx+1:idx+2]
    v = np.zeros(8)
    v[hash(target) % 8] += 1
    for n in neigh:
        v[hash(n) % 8] += 0.5
    return v / (np.linalg.norm(v)+1e-9)

s1 = "he sat by the river bank"
s2 = "she deposited cash at the bank"
e1, e2 = ctx_embed(s1, "bank"), ctx_embed(s2, "bank")
print("cosine between contextual 'bank':", round(float(np.dot(e1, e2)), 3))


## Sentence Transformers

**Definition.** Sentence Transformers fine-tune encoder models with contrastive / triplet / NLI objectives so cosine similarity of pooled embeddings matches semantic similarity.

**Why it matters.** They turn BERT-like encoders into practical retrieval embedders.

**How it works.** Dual encoder with shared weights; train on pairs; at inference mean-pool + normalize; compare with cosine.

**Intuition.** Train the model like a judge of paraphrases, not a fill-in-the-blank exam.

**Common pitfalls.**
- Mismatched train domain vs your corpus
- Forgetting normalization

**When to use.** Default open-source approach for semantic search prototypes and many prod systems.


In [ ]:
# Contrastive loss sketch (educational numbers)
import numpy as np

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

anchor = np.array([0.1, 0.9])
positive = np.array([0.12, 0.88])
negative = np.array([0.9, 0.1])
margin = 0.2
# hinge-style: want cos(a,p) - cos(a,n) > margin
loss = max(0.0, margin - (cosine(anchor, positive) - cosine(anchor, negative)))
print("cos+:", round(cosine(anchor, positive), 3), "cos-:", round(cosine(anchor, negative), 3), "loss:", round(loss, 3))


## Transformer Embeddings

**Definition.** Modern embedding models are transformers specialized for representation: instruction prefixes, Matryoshka dimensions, multilingual & asymmetric towers.

**Why it matters.** State-of-the-art retrieval quality for RAG and enterprise search.

**How it works.** Start from pretrained LM/encoder, fine-tune on large weakly supervised pairs (query↔passage), evaluate on MTEB-style benchmarks, deploy via API or local weights.

**Intuition.** General language understanding + a finishing school for 'find the right passage'.

**Common pitfalls.**
- Chasing leaderboard wins that don't match your domain
- Not re-encoding the corpus after upgrades

**When to use.** Production semantic retrieval unless constraints force classical methods.

| Technique | Contextual? | Retrieval-ready? | Notes |
|---|---|---|---|
| Word2Vec | No | No | Static word table |
| GloVe | No | No | Global co-occurrence |
| FastText | No | Limited | Subwords help OOV |
| BERT raw | Yes | Weak | Needs further fine-tuning |
| Sentence Transformers | Yes | Yes | Great open default |
| Modern emb. APIs | Yes | Yes | Best quality / easiest ops |


In [ ]:
# Instruction-aware embedding formatting examples
examples = [
    ("e5", "query: How to reset MFA?", "passage: To reset MFA, open Security Settings..."),
    ("bge", "Represent this sentence for searching relevant passages: How to reset MFA?", "To reset MFA, open Security Settings..."),
]
for name, q, d in examples:
    print(f"[{name}] Q:", q[:64] + ("..." if len(q)>64 else ""))
    print(f"[{name}] D:", d[:64] + ("..." if len(d)>64 else ""))
    print()


### Try it yourself — Technique selection

1. For a multilingual FAQ with many typos, which classical technique helps most and why?
2. Why might raw BERT underperform a smaller sentence-transformer on FAQ retrieval?
3. Draft an offline re-embedding checklist for upgrading embedding models.


<!-- enriched:v1 -->

### Static vs Contextual Comparison

Static embeddings assign one vector per word type. Contextual models produce different vectors based on sentence context — critical for polysemy.

In [ ]:
# Polysemy illustration with fake contextual shifts
import numpy as np

def fake_contextual(word, sentence, dim=8):
    base = np.random.RandomState(abs(hash(word)) % (2**32)).randn(dim)
    ctx = np.random.RandomState(abs(hash(sentence)) % (2**32)).randn(dim) * 0.3
    v = base + ctx
    return v / np.linalg.norm(v)

w1 = fake_contextual("bank", "river bank erosion")
w2 = fake_contextual("bank", "bank account fees")
print("cosine across senses:", round(float(w1 @ w2), 3), "(lower = more different)")

### From Word2Vec to Sentence Transformers

Pipeline evolution:
1. Word vectors + average pooling
2. Contextual tokens (BERT)
3. Contrastive sentence objectives (SBERT, SimCSE, E5)
4. Instruction-aware embeddings (task prompts / prefixes)

In [ ]:
# Mean-pool BERT-like token matrix vs [CLS]
import numpy as np
tokens = np.random.RandomState(0).randn(10, 12)
cls_vec = tokens[0]
mean_vec = tokens.mean(0)
# In practice SBERT uses mean pooling + normalize more often than raw CLS
for name, v in [("CLS", cls_vec), ("mean", mean_vec)]:
    vn = v / np.linalg.norm(v)
    print(name, np.round(vn[:4], 3))

## Applied Workshop — 04 Embedding Techniques

**Definition.** This workshop converts the ideas in `01-embeddings/04-embedding-techniques.ipynb` into checkable artifacts: a tiny eval, a failure-mode list, and a production checklist.

**Why it matters.** Reading creates familiarity; drills create transferable skill.

**How it works.** Implement the demos below, then adapt them to your domain corpus.

**Intuition.** Treat each notebook like a lab report, not a blog post.

**Common pitfalls.**
- Skipping measurement and relying on a single happy-path example
- Copying thresholds/models without calibration
- Forgetting the online/offline contract (same preprocessing)

**When to use.** After finishing the core sections—before you claim the skill.


In [ ]:
# Workshop harness for 04 embedding techniques
from dataclasses import dataclass

@dataclass
class Check:
    name: str
    ok: bool = False

checks = [
    Check("restated learning objectives"),
    Check("ran two code demos with modified inputs"),
    Check("wrote one domain example"),
    Check("named a metric + failure mode"),
    Check("documented keys/env vars needed"),
]
for c in checks:
    print(f"[{'x' if c.ok else ' '}] {c.name}")


### Try it yourself — Measurement

1. Create three examples: easy, medium, adversarial.
2. Predict outcomes before running code, then compare.
3. Log one metric you would monitor weekly in production.


In [ ]:
# Tiny metric logger
from statistics import mean
trials = {"easy": 1.0, "medium": 0.5, "adversarial": 0.0}
print("mean score", mean(trials.values()))
print("worst case", min(trials, key=trials.get), trials[min(trials, key=trials.get)])


## Decision Table

| If you see... | First try... | Then verify... |
|---|---|---|
| Sudden quality drop | Model/version mismatch | Recompute gold recall |
| High latency | Profile stages | Cache or shrink candidate k |
| Hallucinations in RAG | Retrieval empty/weak | Threshold + refuse path |
| Good demo, bad prod | Distribution shift | Domain eval set |


In [ ]:
# Stage timer sketch
import time
stages = ["retrieve", "rerank", "generate"]
for s in stages:
    t0 = time.perf_counter()
    time.sleep(0.001)
    print(f"{s}: {(time.perf_counter()-t0)*1000:.2f} ms (toy)")


```mermaid
flowchart TB
  A[Learn concept] --> B[Run demo]
  B --> C[Modify inputs]
  C --> D[Measure]
  D --> E{Good enough?}
  E -->|no| F[Adjust + retest]
  F --> D
  E -->|yes| G[Document contract + monitors]
```


### Try it yourself — Teach-back

1. Explain the notebook's core idea to a teammate in 90 seconds.
2. Ask them to spot one risk you missed.
3. Capture both in a short ADR (Architecture Decision Record).


In [ ]:
adr = {
    "title": "Choice for this topic",
    "context": "...",
    "decision": "...",
    "consequences": ["...", "..."],
}
for k, v in adr.items():
    print(f"{k}: {v}")


## Glossary Boost

- **Online path**: request-time compute (query embed, search, generate)
- **Offline path**: batch index build / evaluation jobs
- **Contract**: model + metric + preprocessing agreement
- **Golden set**: labeled examples used as a quality ruler
- **p95**: latency percentile for user-experience budgeting


## Summary & Key Takeaways

- Static word vectors taught distributional semantics but fail on polysemy and retrieval.
- Contextual transformers fixed context; sentence-transformers fixed similarity training.
- Production RAG should use retrieval-tuned embedding models with consistent prefixes.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
